In [373]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import re


In [374]:
user_query = pd.read_csv("dataset_users_query.csv")
answer = pd.read_csv("answers.csv")

In [375]:
answer.drop(["Unnamed: 0"], axis=1, inplace=True)

In [376]:
user_query['query'] = user_query['query'].str.lower()

In [377]:
user_query['query'] = user_query['query'].apply(lambda x: re.sub(r'[^а-яА-ЯёЁa-zA-Z0-9\s]', '', x))
user_query['query'] = user_query['query'].str.strip()


In [378]:
from natasha import Doc, Segmenter, NewsEmbedding, NewsMorphTagger, MorphVocab

segmenter = Segmenter()
morph_tagger = NewsMorphTagger(NewsEmbedding())
morph_vocab = MorphVocab()

def lemmatize_text(text):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
    return ' '.join([token.lemma for token in doc.tokens])

user_query['query_lemmatized'] = user_query['query'].apply(lemmatize_text)



In [379]:
import nltk
from nltk.corpus import stopwords

# Получение стоп-слов 
nltk_stop_words = stopwords.words('russian')
custom_stop_words = ['работа', 'вакансия', 'требуется', 'поиск', 'резюме', 'найти', 'работник', "предлагать", 'скачать']
all_stop_words = set(nltk_stop_words + custom_stop_words)

# Функция для удаления стоп-слов
def remove_stop_words_nltk(text):
    words = text.split()
    words = [word for word in words if word not in all_stop_words]
    return ' '.join(words)

user_query['query_cleaned'] = user_query['query_lemmatized'].apply(remove_stop_words_nltk)
user_query['query_lemmatized'] = user_query['query_cleaned']
user_query.drop(["query_cleaned"], axis=1, inplace=True)
user_query

,query,query_lemmatized
0,фарпост работа владивосток,фарпост владивосток
1,кофейни вакансии,кофейня
2,работа разнорабочие часовой,разнорабочий часовой
3,личный водитель на день,личный водитель день
4,работа от работодателя персональный водитель,работодатель персональный водитель
...,...,...
17365,япония вакансии,япония
17366,япония работа,япония
17367,японский язык работа,японский язык
17368,яппи вакансии,яппи


In [380]:
import re

text_prof = " ААварийный комиссарАвтодиагностАвтомалярАвтоподборщикАвтор студенческих работАвторазборщикАвтослесарьАвтоэкспертАвтоэлектрикАгент по организации обслуживания пассажировАгент по организации почтово-грузовых авиаперевозокАгрономАдминистраторАккумуляторщикАктер, актрисаАкушерАналитикАниматорАппаратчикАрматурщикАрт-директорАртистАрхивариусАрхитекторАссистентАсфальтобетонщикАудиторББалетмейстерБанщикБаристаБарменБетонщикБиблиографБиблиотекарьБилетерБоецБоцманБригадирБурильщикБуфетчикБуфетчик судовойБухгалтерВВальцовщикВальщик леса, лесорубВарщик пищевого сырья и продуктовВахтерВедущий мероприятийВедущий специалист-экспертВесовщикВзрывникВидеографВидеомонтажерВизажистВодительВодолазВоеннослужащий по контрактуВожатыйВожатый служебных собакВокалистВоспитательВрачВыгульщик собакВышивальщица, вышивальщикВязальщикГГабионщикГазорезчикГальваникГардеробщикГеодезистГеологГигиенист стоматологическийГид, экскурсоводГидравлистГидромониторщикГладильщикГорнорабочийГорный мастерГрузчикГрумерГувернантка, гувернерДДворникДежурный по железнодорожной станцииДежурный по переездуДезинфекторДекларант, таможенный брокерДекораторДелопроизводительДетейлерДефектологДефектоскопистДизайнерДиректорДиспетчерДневальныйДозировщикДознавательДокер-механизаторДокмейстерДокументоведДомработница, домработникДонкерманДорожный мастерДорожный рабочийДоярка, доярДробильщикДрожжеводЖЖаровщикЖестянщикЖивотноводЖиловщик мясаЖурналистЗЗаведующийЗагрузчик-выгрузчикЗакальщик мороженогоЗакройщикЗамерщикЗаместительЗаправщик автокондиционеровЗаправщик АЗСЗаправщик картриджейЗаточникЗвукооператорЗвукорежиссерЗемлекопЗооняня, рабочий по уходу за животнымиЗоотехникЗуборезчикИИзготовитель конфетИзготовитель творогаИзолировщикИнженерИнкассаторИнспекторИнспектор службы безопасностиИнспектор транспортной безопасностиИнструкторИнтервьюерИнтернет-маркетологИнъектировщикИхтиопатологККабельщикКазначейКальянщикКаменщикКапитанКассирКастелянша, заведующий бельевойКинологКладовщикКлееварКолористКольщик дровКомендантКомплектовщикКомпрессорщикКондитерКондукторКонсультантКонсьержКонтролерКонюхКоординаторКопирайтерКоптильщикКоренщикКорреспондентКосильщикКосметик-эстетистКосметологКостюмерКочегарКредитный брокерКредитный специалистКредитный экспертКровельщикКрупьеКузнецКузовщикКурьерКухонный работникЛЛаборантЛепщик полуфабрикатовЛесникЛифтерЛогистММалярМаркетологМаркировщикМаркшейдерМассажистМастер аварийно-ремонтной службыМастер буровойМастер депиляцииМастер контрольныйМастер лесозаготовокМастер ногтевого сервисаМастер общестроительных работМастер отделочных работМастер откМастер по добычеМастер по изготовлению РВДМастер по ламинированию ресницМастер по наращиванию волосМастер по наращиванию ресницМастер по обработке камняМастер по обработке рыбыМастер по перетяжке салонов автомобилейМастер по ремонтуМастер по санитарному содержаниюМастер по удалению вмятин без покраскиМастер по эксплуатацииМастер погрузочно-разгрузочных работМастер производстваМастер производственного обученияМастер смрМастер татуировкиМастер участкаМастер цехаМастер-бровистМатросМашинистМедицинский представительМедрегистраторМедсестра, медбратМенеджерМерчендайзерМетеорологМетодистМетодологМетрологМеханизаторМеханикМикробиологМодельМодельерМодераторМойщикМонтажникМонтер путиМонтировщик сценыМотомеханикМотористМузыкальный работникМузыкальный руководительМузыкантМясникННабивщикНаладчикНаучный сотрудникНяняООбвальщикОбивщикОбмотчик элементов электрических машинОвощеводОгнеупорщикОзеленительОклейщикОперативный дежурныйОператорОперационистООперуполномоченныйОптикОрганизаторОсветитель, светотехникОсмотрщик вагоновОтделочникОфициантОформительОхранникОценщикППарикмахерПарковщикПастухПедагогПекарьПереводчикПескоструйщикПечатникПечникПивоварПильщикПлиточникПлотникПоварПодготовитель пищевого сырья и материаловПодносчик багажаПодручный сталевараПожарныйПолиграфологПолировщикПолицейскийПомощникПортнойПочтальонПрачечникПредставитель банкаПреподавательПрессовщикПриемосдатчик груза и багажаПриемщикПровизорПроводник пассажирского вагонаПрограммистПродавецПроектировщикПромоутерПромывальщик геологических пробПромышленный альпинистПрорабПроходчикПсихологПтицеводПтичница, птичникПчеловодРРаботник производственного цехаРаботник торгового залаРабочийРадиомеханикРадиомонтажник судовойРадиооператорРамщикРаскройщикРаспиловщикРаспространительРастениеводРевизорРегистраторРегулировщик радиоэлектронной аппаратурыРегулировщик скорости движения вагоновРедакторРежиссерРезчикРекламный агентРемонтник искусственных сооруженийРентгенолаборантРепетиторРеставраторРефмашинистРитуальный агентРубщик судовойРуководитель, начальникРулевойРыбообработчикССадовникСанитарСборщикСварщикСекретарьСервисный консультантСетевязальщикСигналистСиделкаСистемный администраторСистемотехникСледовательСлесарьСливщик-разливщикСметчикСмотрительСомелье, кавистСопровождающийСортировщикСоставитель поездовСоставитель смесейСоставитель фаршаСотрудник линии раздачиСпециалист абонентского отделаСпециалист аппарата мирового судьиСпециалист АХОСпециалист информационного отделаСпециалист клиентских подключенийСпециалист отдела кадровСпециалист отдела оперативной службыСпециалист отдела розничного бизнесаСпециалист отдела технического сопровожденияСпециалист паспортного столаСпециалист планово-экономического отделаСпециалист по взысканию задолженности / КоллекторСпециалист по воинскому учетуСпециалист по вооружениюСпециалист по госзакупкамСпециалист по гостеприимствуСпециалист по гражданской обороне и чрезвычайным ситуациямСпециалист по замене маслаСпециалист по защите информацииСпециалист по изготовлению мебелиСпециалист по изготовлению наружной рекламыСпециалист по изготовлению стеклопакетовСпециалист по информационной безопасностиСпециалист по коррекции фигурыСпециалист по мобилизационной работеСпециалист по обслуживаниюСпециалист по обслуживанию юридических лицСпециалист по организации мультимодальных перевозокСпециалист по охране трудаСпециалист по пластиковым картамСпециалист по предпродажной подготовкеСпециалист по работе с детьмиСпециалист по работе с населениемСпециалист по работе с семьейСпециалист по развал-схождениюСпециалист по ремонту автостеколСпециалист по ремонту стартеров и генераторовСпециалист по ремонту топливной аппаратурыСпециалист по ремонту трансмиссииСпециалист по ремонту ходовой частиСпециалист по сертификатамСпециалист по системам телефонииСпециалист по согласованиямСпециалист по социальной работеСпециалист по тендерамСпециалист по укладке асфальтаСпециалист по укладке тротуарной плитки, брусчаткиСпециалист по управлению многоквартирными домамиСпециалист по учебно-методической работеСпециалист по химчисткеСпециалист по экономической безопасностиСпециалист сервисного обслуживанияСпециалист службы безопасностиСпециалист строительного контроляСпециалист технической поддержкиСпециалист фрахтового отделаСпециалист-экспертСтаночникСтекольщикСтивидорСтилистСтолярСторожСтраховой агентСтрелокСтяжечникСу-шефСудебный приставСудовой агентСудокорпусник-ремонтникСупервайзерСуперинтендантСюрвейерТТабельщикТайный покупательТакелажникТальманТанцовщица, танцовщикТеплотехникТермистТестировщикТестоводТехникТехник зубнойТехник-лаборантТехник-энергетикТехнический специалистТехнический экспертТехнологТовароведТокарьТонировщикТорговый агентТорговый представительТранспортерщикТранспортировщикТрейдерТренерТрубопроводчик линейныйТрубопроводчик судовойТьюторУУборщик, горничнаяУкладчикУпаковщикУправляющийУстановщикУчастковый уполномоченный полицииУчетчикУчительФФактуровщикФармацевтФасовщикФельдшерФельдъегерьФизикФинансистФинансовый консультантФискаристФлористФормовщикФотографФрезеровщикФутеровщикХХимикХореографХостесХудожникЧЧертежникЧистильщикШШвейцарШвеяШелкографШиномонтажникШишкарьШлаковщикШлифовщикШтукатурЭЭкипировщикЭкологЭкономистЭкспедиторЭксперт по недвижимостиЭлектрикЭлектрогазосварщикЭлектромеханикЭлектромонтажникЭлектромонтерЭлектросварщикЭлектрослесарьЭнергетикЭтикетировщикЮЮвелирЮрисконсультЮрист"
text_busyness = "ДневнаяВечерняяНочнаяПосуточнаяПосменнаяПодработкаУдаленная ДомВахтаВременная ВыходнымНа неполный деньДень"
text_additional = "ИнвалидСтудентШкольникПенсионерМужчинаЖенщина"
text_conditions = "Ежедневная оплатаПроживание"

def text_split(text):
    
    list_t = re.findall(r'[А-Я][^А-Я]*', text)
    filtered_list_t = [prof for prof in list_t if len(prof) > 2]
    cleaned_list_t = []

    for prof in filtered_list_t:
        # Разделяем по запятой и убираем лишние пробелы
        split_prof = [p.strip() for p in prof.split(',')]
        # Добавляем результат в новый список
        cleaned_list_t.extend(split_prof)
    return cleaned_list_t

professions = text_split(text_prof)
employment_terms = text_split(text_busyness)
additional_terms = text_split(text_additional)
conditions_terms =text_split(text_conditions)

professions_cleaned = [a.lower() for a in professions]
employment_terms_cleaned  = [b.lower() for b in employment_terms]
additional_terms_cleaned  = [c.lower() for c in additional_terms]
conditions_terms_cleaned = [d.lower() for d in conditions_terms]

In [381]:
employment_terms_cleaned

['дневная',
 'вечерняя',
 'ночная',
 'посуточная',
 'посменная',
 'подработка',
 'удаленная',
 'дом',
 'вахта',
 'временная',
 'выходным',
 'на неполный день',
 'день']

In [382]:
# Функция для назначения меток
def assign_clusters(query, cluster_terms):
    matched_clusters = []
    for cluster, terms in cluster_terms.items():
        for term in terms:
            if term in query:
                matched_clusters.append(cluster)
                matched_clusters.append(term)
                break
    return matched_clusters if matched_clusters else [0]  # Класс для неопределенных запросов

# Словарь с признаками для каждого кластера
cluster_terms = {
    'Employment': employment_terms_cleaned,
    'Professions': professions_cleaned,
    'Additional': additional_terms_cleaned,
    'Conditions': conditions_terms_cleaned,
}

In [364]:
import pandas as pd
import numpy as np

# Создание нового датафрейма с нужной структурой
result_df = pd.DataFrame(columns=['query', 'занятость', 'по должности-лемме', 'по дополнительному признаку', 'по условиям', 'общие фразы'])

# Функция для заполнения строк в новом датафрейме
def fill_clusters(row):
    query = row['query_lemmatized']
    clusters = assign_clusters(query, cluster_terms)
    ##print(clusters)
    # По умолчанию все значения 0
    occupancy = position_lemma = additional = conditions = 0
    general_phrase = 0

    # Проверка, какие кластеры совпадают с запросом
    if 'Employment' in clusters:
        occupancy = clusters[clusters.index('Employment') + 1]  
    if 'Professions' in clusters:
        position_lemma = clusters[clusters.index('Professions') + 1]  
    if 'Additional' in clusters:
        additional = clusters[clusters.index('Additional') + 1] 
    if 'Conditions' in clusters:
        conditions = clusters[clusters.index('Conditions') + 1] 
    if 0 in clusters:
        general_phrase = 'общая фраза'  

    return pd.Series([query, occupancy, position_lemma, additional, conditions, general_phrase])

# Применение функции к каждому запросу и добавление результатов в новый датафрейм
result_df[['query', 'занятость', 'по должности-лемме', 'по дополнительному признаку', 'по условиям', 'общие фразы']] = user_query.apply(fill_clusters, axis=1)

# Вывод результата
result_df


,query,занятость,по должности-лемме,по дополнительному признаку,по условиям,общие фразы
0,фарпост владивосток,0,0,0,0,общая фраза
1,кофейня,0,0,0,0,общая фраза
2,разнорабочий часовой,0,рабочий,0,0,0
3,личный водитель день,день,водитель,0,0,0
4,работодатель персональный водитель,0,водитель,0,0,0
...,...,...,...,...,...,...
17365,япония,0,0,0,0,общая фраза
17366,япония,0,0,0,0,общая фраза
17367,японский язык,0,0,0,0,общая фраза
17368,яппи,0,0,0,0,общая фраза


In [383]:
answer

,query,занятость,по должности-лемме,по дополнительному признаку,по условиям,общие фразы
0,фарпост работа владивосток,NaN,NaN,NaN,NaN,общая фраза
1,кофейни вакансии,NaN,NaN,NaN,NaN,общая фраза
2,работа разнорабочие часовой,NaN,Рабочий,NaN,NaN,NaN
3,личный водитель на день,на неполный день,Водитель,NaN,NaN,NaN
4,работа от работодателя персональный водитель,NaN,Водитель,NaN,NaN,NaN
...,...,...,...,...,...,...
14240,япония вакансии,NaN,NaN,NaN,NaN,общая фраза
14241,япония работа,NaN,NaN,NaN,NaN,общая фраза
14242,японский язык работа,NaN,NaN,NaN,NaN,общая фраза
14243,яппи вакансии,NaN,NaN,NaN,NaN,общая фраза


In [384]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

target_cols = ['занятость', 'по должности-лемме', 'по дополнительному признаку', 'по условиям', 'общие фразы']
y = result_df[target_cols]

X = result_df['query']
tfidf_vectorizer = TfidfVectorizer()
X_vectorized = tfidf_vectorizer.fit_transform(X)
y

,занятость,по должности-лемме,по дополнительному признаку,по условиям,общие фразы
0,0,0,0,0,общая фраза
1,0,0,0,0,общая фраза
2,0,рабочий,0,0,0
3,день,водитель,0,0,0
4,0,водитель,0,0,0
...,...,...,...,...,...
17365,0,0,0,0,общая фраза
17366,0,0,0,0,общая фраза
17367,0,0,0,0,общая фраза
17368,0,0,0,0,общая фраза


In [385]:
y = y.applymap(lambda x: 1 if x != 0 else 0)
y

,занятость,по должности-лемме,по дополнительному признаку,по условиям,общие фразы
0,0,0,0,0,1
1,0,0,0,0,1
2,0,1,0,0,0
3,1,1,0,0,0
4,0,1,0,0,0
...,...,...,...,...,...
17365,0,0,0,0,1
17366,0,0,0,0,1
17367,0,0,0,0,1
17368,0,0,0,0,1


In [386]:
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)

In [387]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

base_model = RandomForestClassifier(random_state=42)
model = MultiOutputClassifier(base_model)
model.fit(X_train, y_train)


MultiOutputClassifier(estimator=RandomForestClassifier(random_state=42))

In [388]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=target_cols, zero_division=1))
print(classification_report(y_test, y_pred, target_names=target_cols, zero_division=0))

                             precision    recall  f1-score   support

                  занятость       1.00      0.98      0.99       295
         по должности-лемме       1.00      0.97      0.98      1780
по дополнительному признаку       1.00      1.00      1.00       104
                по условиям       1.00      1.00      1.00        46
                общие фразы       0.96      0.99      0.98      1448

                  micro avg       0.98      0.98      0.98      3673
                  macro avg       0.99      0.99      0.99      3673
               weighted avg       0.98      0.98      0.98      3673
                samples avg       0.98      0.98      1.00      3673

                             precision    recall  f1-score   support

                  занятость       1.00      0.98      0.99       295
         по должности-лемме       1.00      0.97      0.98      1780
по дополнительному признаку       1.00      1.00      1.00       104
                по условиям   

In [409]:
test_words = "маляр работа студент"

test_words_vector = tfidf_vectorizer.transform([test_words])

test = model.predict(test_words_vector)
print(test)


[[0 1 1 0 0]]


In [370]:
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier

# Инициализация модели логистической регрессии
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
multi_output_log_reg = MultiOutputClassifier(log_reg_model)

# Обучение модели
multi_output_log_reg.fit(X_train, y_train)

# Предсказание
y_pred_log_reg = multi_output_log_reg.predict(X_test)

# Оценка результатов
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_log_reg, target_names=target_cols, zero_division=1))


                             precision    recall  f1-score   support

                  занятость       1.00      0.93      0.96       295
         по должности-лемме       1.00      0.94      0.97      1780
по дополнительному признаку       1.00      0.88      0.93       104
                по условиям       1.00      0.93      0.97        46
                общие фразы       0.94      0.99      0.97      1448

                  micro avg       0.98      0.96      0.97      3673
                  macro avg       0.99      0.93      0.96      3673
               weighted avg       0.98      0.96      0.97      3673
                samples avg       0.98      0.96      0.99      3673

